# 8. Data and leakage checks
Eight checks for the model info sheet (Appendix D). Each prints PASS or FAIL.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

raw = pd.read_csv("Dataset_Thesis.csv", parse_dates=["Date"]).sort_values(["Ticker", "Date"]).reset_index(drop=True)
### round_trip: read the numbers back exactly as notebook 1 saved them ###
data = pd.read_csv("model_data.csv", parse_dates=["Date"], float_precision="round_trip")

FEATURES_A = ["ret_lag1", "ret_lag2", "ret_lag3", "ret_lag4", "ret_lag5",
              "hl_range", "intraday_ret", "dlog_volume"]
FEATURES_B = FEATURES_A + ["dist_sma10", "dist_sma50", "rsi14", "macd_rel", "macd_signal_rel"]
TECH = ["AAPL", "MSFT", "AMZN", "GOOGL", "TSLA"]
TEST_YEARS = [2020, 2021, 2022, 2023, 2024]

def check(name, passed, detail=""):
    print(("PASS  " if passed else "FAIL  ") + name, detail)

def split(test_year, train_stocks, test_stocks):
    train = data[(data["year"] < test_year) & data["Ticker"].isin(train_stocks)]
    train = train[train["Date"] != train["Date"].max()]
    test = data[(data["year"] == test_year) & data["Ticker"].isin(test_stocks)]
    return train, test

In [2]:
### 1. sample ###
check("27,115 observations, 2,465 per stock, no missing features",
      len(data) == 27115 and data.groupby("Ticker").size().eq(2465).all()
      and data[FEATURES_B].notna().all().all())

### 2. target = next trading day of the same stock ###
next_close = raw.groupby("Ticker")["Adj Close"].shift(-1)
check("target aligned with the next trading day",
      (next_close.isna() | np.isclose(next_close, raw["Next_Day_Adj_Close"])).all()
      and ((raw["Next_Day_Adj_Close"] > raw["Adj Close"]).astype(int) == raw["Target"]).all())

### 3. the most recent return is the return of day t, not t+1 ###
raw["return_today"] = np.log(raw["Adj Close"] / raw.groupby("Ticker")["Adj Close"].shift(1))
merged = data.merge(raw[["Date", "Ticker", "return_today"]], on=["Date", "Ticker"])
check("ret_lag1 is the return of day t", np.allclose(merged["ret_lag1"], merged["return_today"]))

### 4. no single feature is suspiciously predictive ###
single = {f: roc_auc_score(data["Target"], data[f]) for f in FEATURES_B}
strongest = max(single, key=lambda f: abs(single[f] - 0.5))
check("no single feature with |ROC-AUC - 0.5| >= 0.05", abs(single[strongest] - 0.5) < 0.05,
      f"(strongest: {strongest}, {single[strongest]:.3f})")

PASS  27,115 observations, 2,465 per stock, no missing features 
PASS  target aligned with the next trading day 
PASS  ret_lag1 is the return of day t 
PASS  no single feature with |ROC-AUC - 0.5| >= 0.05 (strongest: intraday_ret, 0.486)


In [3]:
### 5. splits: training before the test year, one-day gap, held-out stocks never in training ###
ok = True
for year in TEST_YEARS:
    train, test = split(year, data["Ticker"].unique(), data["Ticker"].unique())
    ok &= (test["Date"].min() - train["Date"].max()).days >= 2
    for stock in TECH:
        train, test = split(year, [s for s in TECH if s != stock], [stock])
        ok &= stock not in set(train["Ticker"])
check("training ends before the test year with a one-day gap; held-out stocks unseen", ok)

### 6. scale-free features: comparable spread across stocks ###
iqr = data.groupby("Ticker")[FEATURES_B].quantile(0.75) - data.groupby("Ticker")[FEATURES_B].quantile(0.25)
ratio = (iqr.max() / iqr.min()).max()
check("feature spread comparable across stocks (IQR ratio < 10)", ratio < 10, f"(largest: {ratio:.1f})")

PASS  training ends before the test year with a one-day gap; held-out stocks unseen 
PASS  feature spread comparable across stocks (IQR ratio < 10) (largest: 4.1)


In [4]:
### 7. shuffled labels: nothing to learn, so test ROC-AUC must be about 0.5 ###
rng = np.random.default_rng(0)
aucs = []
for year in TEST_YEARS:
    train, test = split(year, data["Ticker"].unique(), data["Ticker"].unique())
    model = lgb.LGBMClassifier(n_estimators=200, random_state=0, verbose=-1)
    model.fit(train[FEATURES_B], rng.permutation(train["Target"].to_numpy()))
    aucs.append(roc_auc_score(test["Target"], model.predict_proba(test[FEATURES_B])[:, 1]))
check("shuffled labels give chance-level ROC-AUC", max(abs(np.array(aucs) - 0.5)) < 0.02,
      f"({np.round(aucs, 3).tolist()})")

### 8. positive control: a weak synthetic signal must be found ###
rng = np.random.default_rng(1)
data["weak_signal"] = data["Target"] + rng.normal(0, 4, len(data))
features = FEATURES_B + ["weak_signal"]
train, test = split(2022, data["Ticker"].unique(), data["Ticker"].unique())
fit_part = train[train["year"] < 2021]
fit_part = fit_part[fit_part["Date"] != fit_part["Date"].max()]
val_part = train[train["year"] == 2021]
model = lgb.LGBMClassifier(n_estimators=1000, random_state=0, verbose=-1)
model.fit(fit_part[features], fit_part["Target"], eval_set=[(val_part[features], val_part["Target"])],
          eval_metric="auc", callbacks=[lgb.early_stopping(50, verbose=False)])
model = lgb.LGBMClassifier(n_estimators=max(model.best_iteration_, 1), random_state=0, verbose=-1)
model.fit(train[features], train["Target"])
found = roc_auc_score(test["Target"], model.predict_proba(test[features])[:, 1])
check("positive control: weak synthetic signal recovered", found > 0.53,
      f"(signal alone {roc_auc_score(test['Target'], test['weak_signal']):.3f}, model {found:.3f})")

PASS  shuffled labels give chance-level ROC-AUC ([0.486, 0.495, 0.508, 0.492, 0.501])


PASS  positive control: weak synthetic signal recovered (signal alone 0.581, model 0.570)
